In [1]:
import pickle
import pandas as pd
from sklearn.ensemble import RandomForestClassifier  # If needed, but we'll stick to requested
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

In [2]:
# Load the dataset
df = pd.read_csv("adult.csv")
df.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [3]:
# Replace '?' with None and drop missing values
df = df.replace("?", None)
df = df.dropna()

In [6]:
# drop duplicates
df = df.drop_duplicates(inplace=True)

In [4]:
# Split features (X) and target (y)
X = df.drop(columns=["income"])
y = df["income"]

In [6]:
# Identify columns by data type automatically
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

C:\Users\fight\AppData\Local\Temp\ipykernel_11512\3303656021.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object"]).columns.tolist()


In [7]:
# Split into 80% Train and 20% Test sets cleanly
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
# Scale numeric features
scaler = StandardScaler()
X_train_num = pd.DataFrame(
    scaler.fit_transform(X_train[num_cols]), columns=num_cols
)
X_test_num = pd.DataFrame(scaler.transform(X_test[num_cols]), columns=num_cols)

In [9]:
# One-hot encode categorical features
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
X_train_cat = pd.DataFrame(
    encoder.fit_transform(X_train[cat_cols]),
    columns=encoder.get_feature_names_out(cat_cols),
)
X_test_cat = pd.DataFrame(
    encoder.transform(X_test[cat_cols]),
    columns=encoder.get_feature_names_out(cat_cols),
)

In [10]:
# Merge preprocessed features back together
X_train_clean = pd.concat([X_train_num, X_train_cat], axis=1)
X_test_clean = pd.concat([X_test_num, X_test_cat], axis=1)

In [11]:
# Define the models inside simple pipelines
pipelines = {
    "Logistic Regression": Pipeline([("model", LogisticRegression(max_iter=1000))]),
    "Decision Tree": Pipeline([("model", DecisionTreeClassifier(random_state=42))]),
    "SVC": Pipeline([("model", SVC())]),
    "KNN": Pipeline([("model", KNeighborsClassifier())]),
}

In [12]:
# Dictionary to hold accuracy scores
model_accuracies = {}

In [14]:
# Train and evaluate each model
for name, pipeline in pipelines.items():
    print(f"Training {name}...")
    pipeline.fit(X_train_clean, y_train)

Training Logistic Regression...
Training Decision Tree...
Training SVC...
Training KNN...


In [23]:
# Predict and calculate accuracy
predictions = pipeline.predict(X_test_clean)
accuracy = accuracy_score(y_test, predictions)
model_accuracies[name] = accuracy
print(f"{name} Accuracy: {accuracy:.4f}")

KNN Accuracy: 0.8295


In [19]:
# Compare models
for name, acc in model_accuracies.items():
    print(f"{name}: {acc * 100:.2f}%")

KNN: 82.95%


In [20]:
# Find the best performing model name
best_model_name = max(model_accuracies, key=model_accuracies.get)
best_pipeline = pipelines[best_model_name]

print(f"Best accuracy model : {best_model_name}")

Best accuracy model : KNN


In [21]:
# Save the best  accuracy model as model.pkl using pickle
with open("model.pkl", "wb") as file:
    pickle.dump(best_pipeline, file)